# 04 — Rol 4: Integración y recomendaciones

**Diego Alejandro Sandoval (339271)**

Aquí no se pega una conclusión de cada bloque: se aplican herramientas del Rol 1 (entropía,
información mutua, divergencia KL) directamente sobre las salidas del Rol 2 (zonas) y del
Rol 3 (perfiles de viaje) — resultados que ningún bloque puede producir por sí solo.

In [1]:
import pandas as pd, numpy as np

m  = pd.read_parquet('muestra_features.parquet') if False else pd.read_parquet('muestra_50k.parquet')
m['hour'] = m.pickup_datetime.dt.hour
m['is_weekend'] = (m.pickup_datetime.dt.dayofweek >= 5).astype(int)
r2 = pd.read_parquet('rol2_etiquetas.parquet')
r3 = pd.read_parquet('rol3_etiquetas.parquet')

M = m.merge(r2, on='id').merge(r3, on='id')
RESP = M[[f'r{k}' for k in range(5)]].values
print(f"Tabla maestra: {len(M):,} viajes, {M.shape[1]} columnas")


Tabla maestra: 50,004 viajes, 24 columnas


In [2]:
def H(counts):
    c = np.asarray(counts, float); c = c[c>0]; p = c/c.sum()
    return float(-(p*np.log2(p)).sum())
def H_cond(x, y):
    t = pd.crosstab(y, x).values
    return float(sum(t[i].sum()/t.sum()*H(t[i]) for i in range(t.shape[0])))
def MI(x, y): return H(pd.Series(x).value_counts()) - H_cond(x, y)
def MI_perm(x, y, nperm=200, seed=42):
    rng = np.random.default_rng(seed); obs = MI(x, y)
    null = np.array([MI(x, rng.permutation(np.asarray(y))) for _ in range(nperm)])
    return obs, null.mean(), obs - null.mean()


## Pregunta 1 — ¿Qué explica mejor el tipo de viaje: zona (Rol 2) u hora (Rol 1)?

In [3]:
H_perfil = H(M.perfil.value_counts())
print(f"H(perfil) marginal = {H_perfil:.4f} bits")
for nm, col in [('zona (km5)', 'km5'), ('hora', 'hour')]:
    obs, null, neto = MI_perm(M.perfil, M[col])
    print(f"I({nm}; perfil): neto = {neto:.4f} bits ({neto/H_perfil*100:.2f}% de la incertidumbre del tipo)")


H(perfil) marginal = 1.5988 bits


I(zona (km5); perfil): neto = 0.1389 bits (8.69% de la incertidumbre del tipo)


I(hora; perfil): neto = 0.0199 bits (1.24% de la incertidumbre del tipo)


## Pregunta 2 — ¿El fin de semana cambia el tipo de viaje?

In [4]:
def KL(pc, qc, alpha=0.5):
    idx = pc.index.union(qc.index)
    p = pc.reindex(idx, fill_value=0)+alpha; q = qc.reindex(idx, fill_value=0)+alpha
    p, q = p/p.sum(), q/q.sum()
    return float((p*np.log2(p/q)).sum())

L, W = M[M.is_weekend==0], M[M.is_weekend==1]
kl_perfil = KL(W.perfil.value_counts(), L.perfil.value_counts())
print(f"KL(finde||laboral) en TIPO de viaje: {kl_perfil:.4f} bits")
print("Referencia — Rol 1: KL en HORA = 0.1815 bits | KL en ZONA = 0.0549 bits")
print(f"-> el tipo de viaje del fin de semana es {0.1815/kl_perfil:.0f}x más parecido al laboral que la hora")


KL(finde||laboral) en TIPO de viaje: 0.0037 bits
Referencia — Rol 1: KL en HORA = 0.1815 bits | KL en ZONA = 0.0549 bits
-> el tipo de viaje del fin de semana es 49x más parecido al laboral que la hora


## Pregunta 3 — ¿Qué es el ruido de DBSCAN en términos de tipo de viaje?

In [5]:
ruido_idx = M.ruido.values == 1
mezcla = pd.DataFrame({'ruido': RESP[ruido_idx].mean(0), 'normal': RESP[~ruido_idx].mean(0)},
                       index=[f'P{k}' for k in range(5)])
mezcla['razon'] = mezcla.ruido / mezcla.normal
print(mezcla.round(4))


     ruido  normal   razon
P0  0.0244  0.0280  0.8707
P1  0.3059  0.4218  0.7253
P2  0.4073  0.4046  1.0067
P3  0.2277  0.1275  1.7862
P4  0.0347  0.0181  1.9138


## Puente de 3 patas — la madrugada

In [6]:
madr = M.hour.between(0, 5)
print(f"% ruido DBSCAN:      madrugada {M[madr].ruido.mean()*100:.2f}% | resto {M[~madr].ruido.mean()*100:.2f}%")
print(f"% perfil largo (P3): madrugada {RESP[madr.values,3].mean()*100:.1f}% | resto {RESP[~madr.values,3].mean()*100:.1f}%")
print("Con el Rol 1: la madrugada es también la hora con mayor entropía de zona (~52 zonas efectivas, la más alta del día).")


% ruido DBSCAN:      madrugada 2.87% | resto 0.79%
% perfil largo (P3): madrugada 20.1% | resto 11.9%
Con el Rol 1: la madrugada es también la hora con mayor entropía de zona (~52 zonas efectivas, la más alta del día).


## Cadena aeropuerto — JFK vs. LaGuardia

In [7]:
nombres = {0:'Midtown', 1:'Downtown', 2:'Upper Manhattan', 3:'LaGuardia', 4:'JFK'}
for zn in [4, 3]:
    s = M[M.km5 == zn]
    idx = (M.km5==zn).values
    print(f"{nombres[zn]:16s}: {len(s):4d} viajes | ruido {s.ruido.mean()*100:5.2f}% | "
          + " ".join(f"P{k}:{RESP[idx,k].mean()*100:4.1f}%" for k in range(5)))


JFK             : 1085 viajes | ruido  5.07% | P0: 0.6% P1: 1.9% P2: 6.1% P3:41.9% P4:49.5%
LaGuardia       : 1532 viajes | ruido 10.31% | P0: 0.7% P1: 7.4% P2:23.2% P3:68.4% P4: 0.3%


## Recomendaciones operativas cuantificadas

**1. Posicionamiento por zona, no por hora.** La zona explica el tipo de viaje varias veces
mejor que la hora — dimensionar la flota base por zona, y usar la hora solo para escalar
volumen.

**2. Un solo mapa, dos calendarios de turnos.** El fin de semana casi no cambia el tipo de
viaje pero sí la hora (decenas de veces más) — mismo mapa de zonas, calendario de turnos
distinto para laboral y fin de semana/festivos.

**3. Madrugada: despacho bajo demanda, no posicionamiento fijo.** Es la franja más incierta
en las tres dimensiones (zona, ruido, tipo) a la vez.

**4. Tratar JFK y LaGuardia de forma distinta.** JFK tiene una ruta con identidad propia y
predecible; LaGuardia es demanda de viaje largo genérico y más ruidosa.

Los números exactos de cada recomendación están en `docs/documento_soporte.pdf`, sección 8.